# Skin Disease Chatbot using NLP and Machine Learning

**Project Type:** Academic / College Project  
**Objective:** Build a simple chatbot that takes symptom descriptions in natural language and predicts the most likely skin disease category.  

> **DISCLAIMER:** This project is for educational and informational purposes only. It does not provide medical diagnoses. Users should consult a dermatologist for proper diagnosis.

---

## Table of Contents
1. Data Loading & Exploration
2. Data Cleaning
3. Data Preprocessing
4. Feature Extraction (TF-IDF)
5. Model Training & Comparison
6. Model Evaluation
7. Prediction Demo
8. Save Model

## 1. Import Libraries

In [ ]:
import re
import pickle
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.naive_bayes import MultinomialNB
from sklearn.svm import LinearSVC
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    accuracy_score,
    ConfusionMatrixDisplay,
)

import nltk
nltk.download('punkt', quiet=True)
nltk.download('punkt_tab', quiet=True)
nltk.download('stopwords', quiet=True)
nltk.download('wordnet', quiet=True)

from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

import warnings
warnings.filterwarnings('ignore')

print('All libraries imported successfully!')

## 2. Load & Explore Raw Data

In [ ]:
# Load the raw dataset
df = pd.read_csv('../data/raw/skin_disease_symptoms.csv')

print(f'Dataset shape: {df.shape}')
print(f'Columns: {list(df.columns)}')
df.head(10)

In [ ]:
# Data types and info
df.info()

In [ ]:
# Check for missing values
print('Missing values per column:')
print(df.isnull().sum())
print(f'\nTotal missing: {df.isnull().sum().sum()}')

In [ ]:
# Disease distribution
print('Disease class distribution:')
print(df['disease'].value_counts())
print(f'\nNumber of unique diseases: {df["disease"].nunique()}')

In [ ]:
# Visualize disease distribution
fig, ax = plt.subplots(figsize=(10, 5))
df['disease'].value_counts().plot(kind='bar', ax=ax, color='steelblue', edgecolor='black')
ax.set_title('Number of Samples per Disease Class', fontsize=14)
ax.set_xlabel('Disease', fontsize=12)
ax.set_ylabel('Count', fontsize=12)
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

In [ ]:
# Text length analysis
df['text_length'] = df['text'].apply(len)
df['word_count'] = df['text'].apply(lambda x: len(x.split()))

print('Text length statistics:')
print(df[['text_length', 'word_count']].describe())

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
df['text_length'].hist(bins=20, ax=axes[0], color='steelblue', edgecolor='black')
axes[0].set_title('Distribution of Text Length (characters)')
axes[0].set_xlabel('Character Count')

df['word_count'].hist(bins=15, ax=axes[1], color='coral', edgecolor='black')
axes[1].set_title('Distribution of Word Count')
axes[1].set_xlabel('Word Count')
plt.tight_layout()
plt.show()

df.drop(columns=['text_length', 'word_count'], inplace=True)

In [ ]:
# Sample entries per disease
for disease in df['disease'].unique()[:3]:
    print(f'\n--- {disease.upper()} ---')
    samples = df[df['disease'] == disease]['text'].head(3).tolist()
    for s in samples:
        print(f'  - {s}')

## 3. Data Cleaning

In [ ]:
# Remove duplicates
before = len(df)
df = df.drop_duplicates(subset=['text', 'disease'])
print(f'Removed {before - len(df)} duplicate rows')

# Remove rows with missing text or disease
df = df.dropna(subset=['text', 'disease'])
df['text'] = df['text'].astype(str).str.strip()
df['disease'] = df['disease'].astype(str).str.strip()
df = df[df['text'].str.len() > 0]
print(f'After cleaning: {len(df)} rows')

In [ ]:
# Standardize disease labels
LABEL_MAP = {
    'acne': 'acne',
    'eczema': 'eczema',
    'atopic dermatitis': 'eczema',
    'psoriasis': 'psoriasis',
    'ringworm': 'ringworm',
    'tinea': 'ringworm',
    'vitiligo': 'vitiligo',
    'rosacea': 'rosacea',
    'contact dermatitis': 'contact_dermatitis',
    'contact_dermatitis': 'contact_dermatitis',
    'hives': 'hives',
    'urticaria': 'hives',
    'seborrheic dermatitis': 'seborrheic_dermatitis',
    'seborrheic_dermatitis': 'seborrheic_dermatitis',
    'dandruff': 'seborrheic_dermatitis',
    'scabies': 'scabies',
}

df['disease'] = df['disease'].str.lower().str.strip()
df['disease'] = df['disease'].map(LABEL_MAP).fillna(df['disease'])

print('Standardized disease labels:')
print(df['disease'].value_counts())

## 4. Text Preprocessing

### What we apply:
- Lowercase conversion
- Remove URLs, emails, special characters
- Remove extra whitespace
- Tokenization
- Conservative stopword removal (keeps negation words)
- Lemmatization

### What we do NOT apply (to preserve medical info):
- Do NOT remove negation words ("not itchy" vs "itchy")
- Do NOT remove color words ("red", "white", "yellow")
- Do NOT remove body part names
- Do NOT use aggressive stemming

In [ ]:
def clean_text(text):
    """Clean raw text: lowercase, remove special chars, collapse spaces."""
    text = text.lower()
    text = re.sub(r'http\S+|www\.\S+', '', text)
    text = re.sub(r'\S+@\S+', '', text)
    text = re.sub(r'[^a-z0-9\s\-]', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

# Words to KEEP even though standard stopword lists remove them
KEEP_WORDS = {
    'not', 'no', 'nor', 'never', 'without',
    'very', 'more', 'most', 'much',
    'few', 'all', 'both', 'each', 'every',
    'after', 'before', 'during', 'between',
    'under', 'over', 'above', 'below',
}

stop_words = set(stopwords.words('english')) - KEEP_WORDS
lemmatizer = WordNetLemmatizer()

def preprocess_text(text):
    """Full preprocessing: clean -> tokenize -> remove stopwords -> lemmatize."""
    text = clean_text(text)
    tokens = word_tokenize(text)
    tokens = [lemmatizer.lemmatize(t) for t in tokens if t not in stop_words and len(t) > 1]
    return ' '.join(tokens)

print('Preprocessing functions defined.')

In [ ]:
# Apply preprocessing
df['cleaned_text'] = df['text'].apply(clean_text)
df['processed_text'] = df['text'].apply(preprocess_text)

# Remove very short entries
df['word_count'] = df['processed_text'].apply(lambda x: len(x.split()))
df = df[df['word_count'] >= 3]
df.drop(columns=['word_count'], inplace=True)

print(f'After preprocessing: {len(df)} rows')
print(f'\nRaw vs Processed examples:')
for i in range(5):
    print(f'\n  Raw:       {df.iloc[i]["text"]}')
    print(f'  Processed: {df.iloc[i]["processed_text"]}')
    print(f'  Disease:   {df.iloc[i]["disease"]}')

## 5. Encode Labels & Split Data

In [ ]:
# Encode disease labels as integers
le = LabelEncoder()
df['label'] = le.fit_transform(df['disease'])

label_names = dict(zip(le.transform(le.classes_), le.classes_))
print('Label encoding:')
for idx, name in sorted(label_names.items()):
    count = (df['label'] == idx).sum()
    print(f'  {idx}: {name} ({count} samples)')

In [ ]:
# Split into train and test sets
X = df['processed_text'].values
y = df['label'].values

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f'Training set: {len(X_train)} samples')
print(f'Test set:     {len(X_test)} samples')

# Verify stratification
print(f'\nTrain label distribution:')
unique, counts = np.unique(y_train, return_counts=True)
for u, c in zip(unique, counts):
    print(f'  {label_names[u]}: {c}')

## 6. Feature Extraction - TF-IDF

TF-IDF (Term Frequency - Inverse Document Frequency) converts text into numerical vectors:
- **TF**: How often a word appears in a document
- **IDF**: How rare a word is across all documents
- Words that are frequent in one document but rare overall get higher scores

In [ ]:
# Create TF-IDF vectorizer
tfidf = TfidfVectorizer(
    max_features=5000,
    ngram_range=(1, 2),   # unigrams + bigrams
    sublinear_tf=True,    # apply log normalization
)

X_train_tfidf = tfidf.fit_transform(X_train)
X_test_tfidf = tfidf.transform(X_test)

print(f'TF-IDF matrix shape (train): {X_train_tfidf.shape}')
print(f'TF-IDF matrix shape (test):  {X_test_tfidf.shape}')
print(f'Vocabulary size: {len(tfidf.vocabulary_)}')

In [ ]:
# Show top TF-IDF features
feature_names = tfidf.get_feature_names_out()
print(f'Total features: {len(feature_names)}')
print(f'\nSample features (first 30):')
print(feature_names[:30])

## 7. Model Training & Comparison

We train and compare 4 classifiers:
1. **Multinomial Naive Bayes** - fast, works well with TF-IDF
2. **Linear SVM** - strong for text classification
3. **Logistic Regression** - interpretable, reliable
4. **Random Forest** - ensemble method, handles non-linearity

In [ ]:
# Define models
models = {
    'Naive Bayes': MultinomialNB(alpha=0.1),
    'Linear SVM': LinearSVC(max_iter=10000, C=1.0),
    'Logistic Regression': LogisticRegression(max_iter=2000, C=30.0),
    'Random Forest': RandomForestClassifier(n_estimators=200, random_state=42),
}

results = {}

for name, model in models.items():
    print(f'\n{"="*50}')
    print(f'Training: {name}')
    print(f'{"="*50}')
    
    # Cross-validation on training set
    cv_scores = cross_val_score(model, X_train_tfidf, y_train, cv=5, scoring='accuracy')
    print(f'CV Accuracy: {cv_scores.mean():.4f} (+/- {cv_scores.std():.4f})')
    
    # Train on full training set
    model.fit(X_train_tfidf, y_train)
    
    # Predict on test set
    y_pred = model.predict(X_test_tfidf)
    test_acc = accuracy_score(y_test, y_pred)
    print(f'Test Accuracy: {test_acc:.4f}')
    
    results[name] = {
        'model': model,
        'cv_mean': cv_scores.mean(),
        'cv_std': cv_scores.std(),
        'test_acc': test_acc,
        'y_pred': y_pred,
    }

In [ ]:
# Compare all models
comparison = pd.DataFrame({
    'Model': list(results.keys()),
    'CV Accuracy': [r['cv_mean'] for r in results.values()],
    'CV Std': [r['cv_std'] for r in results.values()],
    'Test Accuracy': [r['test_acc'] for r in results.values()],
}).sort_values('Test Accuracy', ascending=False)

print('Model Comparison:')
print(comparison.to_string(index=False))

# Visualize
fig, ax = plt.subplots(figsize=(8, 5))
x = range(len(comparison))
ax.bar(x, comparison['Test Accuracy'], color='steelblue', edgecolor='black', label='Test Accuracy')
ax.bar(x, comparison['CV Accuracy'], color='coral', edgecolor='black', alpha=0.6, label='CV Accuracy')
ax.set_xticks(x)
ax.set_xticklabels(comparison['Model'], rotation=30, ha='right')
ax.set_ylabel('Accuracy')
ax.set_title('Model Comparison')
ax.legend()
ax.set_ylim(0, 1.1)
for i, v in enumerate(comparison['Test Accuracy']):
    ax.text(i, v + 0.02, f'{v:.2f}', ha='center', fontweight='bold')
plt.tight_layout()
plt.show()

## 8. Detailed Evaluation of Best Model

In [ ]:
# Select best model
# Only models with predict_proba can drive follow-up questions in the chatbot
eligible = comparison[comparison['Model'].map(lambda m: hasattr(results[m]['model'], 'predict_proba'))]
best_name = eligible.iloc[0]['Model']
best_result = results[best_name]
best_model = best_result['model']
y_pred_best = best_result['y_pred']

print(f'Best Model: {best_name}')
print(f'Test Accuracy: {best_result["test_acc"]:.4f}')

# Classification report
target_names = [label_names[i] for i in sorted(label_names.keys())]
print(f'\nClassification Report:')
print(classification_report(y_test, y_pred_best, target_names=target_names))

In [ ]:
# Confusion matrix
cm = confusion_matrix(y_test, y_pred_best)

fig, ax = plt.subplots(figsize=(12, 10))
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=target_names)
disp.plot(ax=ax, cmap='Blues', xticks_rotation=45)
ax.set_title(f'Confusion Matrix - {best_name}', fontsize=14)
plt.tight_layout()
plt.show()

## 9. Prediction Demo

Let's test the model with some sample symptom descriptions.

In [ ]:
# Build a full pipeline (TF-IDF + best classifier) for easy prediction
best_pipeline = Pipeline([
    ('tfidf', TfidfVectorizer(max_features=5000, ngram_range=(1, 2), sublinear_tf=True)),
    ('clf', type(best_model)(**best_model.get_params())),
])

# Retrain the pipeline on all training data
best_pipeline.fit(X_train, y_train)

# Verify it matches
pipeline_acc = accuracy_score(y_test, best_pipeline.predict(X_test))
print(f'Pipeline test accuracy: {pipeline_acc:.4f}')

In [ ]:
# Disease information for chatbot responses
DISEASE_INFO = {
    'acne': 'Acne is a common skin condition where hair follicles become clogged with oil and dead skin cells, causing pimples, blackheads, or whiteheads.',
    'eczema': 'Eczema (Atopic Dermatitis) causes dry, itchy, and inflamed skin. It is common in children but can occur at any age.',
    'psoriasis': 'Psoriasis causes red, scaly patches on the skin, often on the elbows, knees, and scalp. It is an autoimmune condition.',
    'ringworm': 'Ringworm is a fungal infection that causes a circular, ring-shaped rash. Despite its name, it is not caused by a worm.',
    'vitiligo': 'Vitiligo causes loss of skin pigment in patches. The affected areas appear white or lighter than the surrounding skin.',
    'rosacea': 'Rosacea causes redness and visible blood vessels on the face, often with small red bumps. It commonly affects the nose and cheeks.',
    'contact_dermatitis': 'Contact Dermatitis is a rash caused by direct contact with an irritant or allergen, such as soaps, metals, or plants.',
    'hives': 'Hives (Urticaria) are raised, itchy welts that appear suddenly, often due to an allergic reaction. They usually fade within hours.',
    'seborrheic_dermatitis': 'Seborrheic Dermatitis causes scaly, flaky, itchy patches on oily areas like the scalp, face, and chest. Dandruff is a mild form.',
    'scabies': 'Scabies is caused by tiny mites that burrow into the skin, causing intense itching especially at night. It spreads through close contact.',
}

def predict_disease(symptom_text):
    """Predict disease from symptom description and return chatbot response."""
    processed = preprocess_text(symptom_text)
    prediction = best_pipeline.predict([processed])[0]
    disease = label_names[prediction]
    info = DISEASE_INFO.get(disease, 'No additional information available.')
    
    print(f'You said: "{symptom_text}"')
    print(f'\nPredicted condition: {disease.replace("_", " ").title()}')
    print(f'\nInfo: {info}')
    print(f'\n** DISCLAIMER: This is for informational purposes only.')
    print(f'   Please consult a dermatologist for proper diagnosis. **')
    print('-' * 60)
    return disease

In [ ]:
# Test with sample inputs
test_symptoms = [
    'I have itchy red patches on my elbows',
    'small pimples and oily skin on my face',
    'circular ring shaped rash on my arm',
    'white patches on my skin losing color',
    'my face is very red with visible blood vessels on cheeks',
    'I got a rash after wearing a nickel necklace',
    'red raised welts appeared after eating shellfish',
    'flaky greasy dandruff on my scalp that wont go away',
    'intense itching between my fingers at night',
    'dry cracked itchy skin behind my knees',
]

for symptom in test_symptoms:
    predict_disease(symptom)
    print()

## 10. Save the Model

In [ ]:
import os
os.makedirs('../models', exist_ok=True)

# Save the pipeline
with open('../models/skin_disease_model.pkl', 'wb') as f:
    pickle.dump(best_pipeline, f)

# Save label names
with open('../models/label_names.pkl', 'wb') as f:
    pickle.dump(label_names, f)

# Save disease info
with open('../models/disease_info.pkl', 'wb') as f:
    pickle.dump(DISEASE_INFO, f)

print('Model saved to ../models/')
print('Files: skin_disease_model.pkl, label_names.pkl, disease_info.pkl')

In [ ]:
# Verify saved model works
with open('../models/skin_disease_model.pkl', 'rb') as f:
    loaded_pipeline = pickle.load(f)
with open('../models/label_names.pkl', 'rb') as f:
    loaded_labels = pickle.load(f)

test_text = 'itchy red scaly patches on my knees'
processed = preprocess_text(test_text)
pred = loaded_pipeline.predict([processed])[0]
print(f'Test: "{test_text}"')
print(f'Prediction: {loaded_labels[pred]}')
print('\nModel loads and predicts correctly!')

## 11. Interactive Chatbot (Run in Notebook)

Run the cell below to interactively type symptoms and get predictions.  
Type `quit` to exit.

In [ ]:
print('=== Skin Disease Chatbot ===')
print('Describe your skin symptoms and I will suggest a possible condition.')
print('Type "quit" to exit.\n')
print('DISCLAIMER: This is for informational purposes only.')
print('Please consult a dermatologist for proper diagnosis.\n')

while True:
    user_input = input('Your symptoms: ')
    if user_input.lower().strip() in ('quit', 'exit', 'q'):
        print('Thank you! Remember to consult a dermatologist for any skin concerns.')
        break
    if len(user_input.strip()) < 5:
        print('Please describe your symptoms in more detail.\n')
        continue
    print()
    predict_disease(user_input)
    print()

---

## Summary

| Component | Details |
|---|---|
| **Dataset** | 501 synthetic symptom descriptions |
| **Classes** | 10 skin diseases (50 samples each) |
| **Preprocessing** | Lowercase, clean, tokenize, lemmatize, conservative stopwords |
| **Features** | TF-IDF (unigrams + bigrams, max 5000 features) |
| **Models tested** | Naive Bayes, Linear SVM, Logistic Regression, Random Forest |
| **Evaluation** | 5-fold CV + held-out test set |
| **Output** | Saved pickle model for deployment |

> **Next Step:** Build a web interface using Flask or Streamlit to deploy this chatbot.